# Day 5 — Extra exercises: regression, prediction and model checking

**For two kinds of reader.** If you **missed Day 5** (after informing the lecturer in advance), this notebook is part of your make-up work. If you attended and want **more practice**, work through any part at your own pace; nothing here is required for you.

**What a make-up submission for Day 5 contains** (see `../day-5-makeup.md`):

- **A1–A3 and B1–B5** of this notebook;
- your own solutions to the core in-class exercises in `../day-5-exercises-morning.ipynb` and `../day-5-exercises-afternoon.ipynb`.

The reflection prompts are optional. **C-problems are optional challenges** for everyone. The make-up submission is graded on the 0–4 in-class scale and counts as your in-class score for the missed day.

**Topics.** Part A: regression as a model that generates data; centring the predictor; priors and prior predictive simulation; writing the regression in Stan. Part B: fitting and reading the regression; the mean versus one new observation; a posterior predictive check; revising the model; an unusual observation and Student-t errors.

**Background.** Read the Day 5 slides first: `../day-5-slides-morning.pdf` (regression as a generative model, priors, prior predictive simulation) and `../day-5-slides-afternoon.pdf` (fit, predict, check, revise). Each task below names the slides it builds on. The demos in `../demos/` show the same steps on the slides' example.

**How to work.** Open in JupyterLab, select **R (bayes)**, run the cells in order from a fresh kernel, from this `extra/` folder. Hints are hidden under **Hint**; try first. Write explanations in your own words in the *Your answer* cells: numbers alone are not enough. All data are **synthetic teaching data**; never report them as real measurements.

**Stan.** Every fit runs live. `models/commute.stan` and `models/commute-growing.stan` are **templates**: complete the lines marked `TODO` (open them in JupyterLab's text editor) before fitting. `models/commute-t.stan` is complete. If your installation fails, use UCloud (see the course page) and tell the lecturer early rather than at the deadline.

**Submitting (make-up).** Submit this notebook with outputs, the `.stan` files you edited, and your written answers, together with the in-class exercises, through the announced route by the announced deadline (normally within one week of the missed day). Discussing ideas is fine; the code and explanations must be your own. You may use AI assistants without declaring it, but you should be able to explain every line.

## Setup

Run this cell first.

In [ ]:
options(tidyverse.quiet = TRUE)
library(tidyverse)   # dplyr for summaries, ggplot2 for figures
library(cmdstanr)    # Stan from R
set.seed(2026)
options(width = 100, repr.plot.width = 8, repr.plot.height = 5, repr.plot.bg = "white")
theme_set(theme_minimal(base_size = 14) + theme(legend.position = "top"))
DATA <- "data"; MODELS <- "models"     # run from the extra/ folder

## The data

A cyclist timed her commute on **five routes** of 2, 4, 6, 8 and 10 km, **eight trips** on each, 40 trips in total (`data/commute-times.csv`: `trip`, `route_km`, `minutes`). She wants to know how her time grows with distance, and what to expect on a 15-km route she has never ridden.

In [ ]:
d <- read_csv(file.path(DATA, "commute-times.csv"), show_col_types = FALSE) |>
  mutate(x = route_km - 6)        # centred predictor: 0 on the 6-km route
head(d)

## Part A — Regression as a model that generates data

### A1. Plot, and choose a scale for x

*Slides "Does her clock scale?", "Regression: a model that generates the data" and "Choose a scale for x" (morning).* The model, with the centred predictor xᵢ = kmᵢ − 6:

yᵢ ~ Normal(α + β · xᵢ, σ), independent given x.

1. Plot commute time against route length, and make a table of the number of trips, mean and SD of the time on each route.
2. In one sentence each, with units: what do α, β and σ mean on this scale? What would α mean with x = km?

In [ ]:
# d |> group_by(route_km) |> summarise(...)
# ggplot(d, aes(x = route_km, y = minutes)) + geom_jitter(width = 0.15, height = 0) + ...

<details><summary><b>Hint</b> (click to open)</summary>

`geom_jitter(width = 0.15, height = 0)` spreads the points sideways a little so that the eight trips on a route do not hide each other, without changing their times.

</details>

**Your answer:** *(write here)*

### A2. Priors and prior predictive simulation

*Slides "Priors on that scale", "Prior predictive simulation" and "What the prior predicts" (morning).* Proposed priors:

- α ~ Normal(20, 5): a 6-km commute by bike takes about 20 minutes, probably 10–30;
- β ~ Normal(3, 1): about 3 minutes per km (20 km/h), probably 1–5;
- σ ~ Normal⁺(0, 5): single trips scatter by less than about 10 minutes.

1. Explain in one sentence why each prior is reasonable (or not), using what you know about cycling.
2. Simulate S = 4000 prior draws and one trip per draw at 2, 6, 10 and 15 km. Give the 90% interval of the simulated time at each distance and the share of negative times.
3. Plot 100 prior mean lines for these priors and for a vague prior α ~ Normal(0, 100), β ~ Normal(0, 10), side by side. What share of draws has a negative slope under each prior, and why does that matter?

In [ ]:
S <- 4000
# prior <- tibble(alpha = rnorm(S, 20, 5), beta = rnorm(S, 3, 1), sigma = abs(rnorm(S, 0, 5)))
# prior |> cross_join(tibble(km = c(2, 6, 10, 15))) |> mutate(y_sim = rnorm(n(), alpha + beta * (km - 6), sigma)) |> ...

<details><summary><b>Hint</b> (click to open)</summary>

Follow the code on the slide "Prior predictive simulation": draw the three parameters, then use `cross_join()` with a small tibble of distances to get one row per draw and distance, then `rnorm(n(), mean, sd)` for the simulated trips. Remember that the mean uses the **centred** distance km − 6.

</details>

**Your answer:** *(write here)*

### A3. Fake data, and the regression in Stan

*Slides "Fake data from the prior" and "The regression in Stan" (morning).*

1. Simulate four fake datasets from the prior at her 40 routes (one prior draw per dataset) and plot them in four panels. Do they look like commutes?
2. Complete the template `models/commute.stan`: the three priors, the likelihood, and in `generated quantities` the mean time `mu_15` on a 15-km route and one simulated trip `y_15` there. Compile it.

In [ ]:
# fake <- tibble(dataset = paste("fake dataset", 1:4), alpha = ..., beta = ..., sigma = ...) |>
#   cross_join(d |> select(route_km, x)) |>
#   mutate(y_sim = rnorm(n(), alpha + beta * x, sigma))

# after completing the TODO lines:
# mod <- cmdstan_model(file.path(MODELS, "commute.stan"))

<details><summary><b>Hint</b> (click to open)</summary>

The likelihood is one line, `y ~ normal(alpha + beta * x, sigma);`. For 15 km the centred predictor is x = 15 − 6 = 9, so `mu_15 = alpha + beta * 9`. One trip: `normal_rng(mu_15, sigma)`.

</details>

**Your answer:** *(write here)*

## Part B — Fit, predict, check

### B1. Fit and read the regression

*Slides "The fit" (afternoon) and Day 3 "Six numbers to check after every fit".*

1. Fit `models/commute.stan` (4 chains, `seed = 2026`) and check the diagnostics.
2. Report α, β and σ with 90% intervals and units, and say what each means for her.
3. Her average speed in km/h is 60 / β. Compute its posterior mean and 90% interval from the draws, and P(speed > 20 km/h | y).

In [ ]:
stan_data <- list(N = nrow(d), x = d$x, y = d$minutes)
# fit <- mod$sample(data = stan_data, chains = 4, seed = 2026, refresh = 0)
# draws <- fit$draws(format = "df")
# draws <- draws |> mutate(speed = 60 / beta)

<details><summary><b>Hint</b> (click to open)</summary>

Any function of the parameters can be computed draw by draw: a new column `speed = 60 / beta` gives draws of her speed, and its mean, quantiles and probabilities follow as for any other column.

</details>

**Your answer:** *(write here)*

### B2. A 15-km route: the mean versus one trip

*Slides "At 20 s: the mean versus one attempt" and "Prediction across all targets" (afternoon).*

1. Give the 90% intervals of `mu_15` and `y_15`, and plot both histograms in one figure with a legend.
2. She must arrive at a meeting 60 minutes after leaving. What is the probability that **one trip** on the 15-km route takes longer than 60 minutes? Why is P(μ₁₅ > 60) the wrong number for her question?
3. What assumptions does this prediction rest on, given that her longest route is 10 km?

In [ ]:
# quantile(draws$mu_15, c(0.05, 0.95));  quantile(draws$y_15, c(0.05, 0.95))
# mean(draws$y_15 > 60)

**Your answer:** *(write here)*

### B3. A posterior predictive check: the spread on each route

*Slides "Posterior predictive checks", "A check: the spread at each target" and "The residuals show it too" (afternoon).* Run the helper cell, then:

1. Run the check with `replicated_sd(draws)`, `ppc_table()` and `ppc_plot()`. On which routes is the observed SD unusual?
2. Plot the residuals (time − fitted mean, with the posterior means of α and β) against route length. Which pattern do you see, and why is it plausible for commuting?

In [ ]:
# Helper for B3 and B4 (run it; you do not need to change it).
# For every posterior draw, replicate all 40 trips on the same routes,
# then take the SD of the replicated trips on each route.
# growing = TRUE: the revised model, where the spread on a route of km kilometres is sigma * km / 6.
replicated_sd <- function(draws, growing = FALSE) {
  as_tibble(draws) |>
    select(.draw, alpha, beta, sigma) |>
    cross_join(d |> select(route_km, x)) |>
    mutate(spread = if (growing) sigma * route_km / 6 else sigma,
           y_rep  = rnorm(n(), alpha + beta * x, spread)) |>
    group_by(.draw, route_km) |>
    summarise(rep_sd = sd(y_rep), .groups = "drop")
}
observed_sd <- d |> group_by(route_km) |> summarise(obs_sd = sd(minutes))

ppc_plot <- function(rep) {
  ggplot(rep, aes(x = rep_sd)) +
    geom_histogram(aes(fill = "replicated datasets"), bins = 40, alpha = 0.7) +
    geom_vline(data = observed_sd, aes(xintercept = obs_sd, colour = "observed data"), linewidth = 1.3) +
    facet_wrap(~ route_km, labeller = label_both, nrow = 1) +
    scale_fill_manual(values = c("replicated datasets" = "grey60"), name = NULL) +
    scale_colour_manual(values = c("observed data" = "tomato"), name = NULL) +
    labs(x = "SD of the trips on this route (minutes)", y = "number of draws")
}
ppc_table <- function(rep) {
  rep |> group_by(route_km) |>
    summarise(rep_q5 = quantile(rep_sd, 0.05), rep_q95 = quantile(rep_sd, 0.95)) |>
    left_join(observed_sd, by = "route_km")
}

In [ ]:
# rep_constant <- replicated_sd(draws)
# ppc_table(rep_constant);  ppc_plot(rep_constant)

**Your answer:** *(write here)*

### B4. Revise the model and check again

*Slide "Revise: the spread grows with the target" (afternoon).*

1. Complete `models/commute-growing.stan`: the spread on a route of km kilometres should be σ · km / 6, so that σ is the spread on the 6-km route. Change the likelihood and `y_15` accordingly.
2. Fit it, check the diagnostics, and run the check with `replicated_sd(draws_growing, growing = TRUE)`.
3. Compare the two models: β, the 90% interval of one 15-km trip, and P(one trip > 60 minutes). Which would you report?

In [ ]:
# fit_growing <- cmdstan_model(file.path(MODELS, "commute-growing.stan"))$sample(data = stan_data, chains = 4, seed = 2026, refresh = 0)
# draws_growing <- fit_growing$draws(format = "df")

<details><summary><b>Hint</b> (click to open)</summary>

Since km = x + 6, the spread is `sigma * (x + 6) / 6`; this replaces `sigma` as the third argument of `normal()`. On the 15-km route the spread is `sigma * 15 / 6`.

</details>

**Your answer:** *(write here)*

### B5. A flat tyre

*Slides "An unusual attempt" and "Robust regression: Student-t errors" (afternoon).* On one more 10-km trip she had a flat tyre and recorded **55 minutes**.

1. Add this trip to the data and fit `models/commute.stan` and `models/commute-t.stan` to the 41 trips. Compare β and σ with the fit to the original 40 trips.
2. Which model would you use for her 15-km question, and what would you do with the flat-tyre trip? (There is no single right answer; argue for your choice.)

In [ ]:
# d_flat <- bind_rows(d, tibble(trip = 41, route_km = 10, minutes = 55)) |> mutate(x = route_km - 6)
# data_flat <- list(N = nrow(d_flat), x = d_flat$x, y = d_flat$minutes)

**Your answer:** *(write here)*

**Reflection (optional).** In 80–120 words: describe the modelling workflow you followed in Parts A and B (*slide "The modelling workflow"*), and say at which step you learned the most about her commute.

**Your reflection:** *(write here)*

## Part C — Optional challenges

### C1. How much does the prior matter with little data?

Keep only her **first trip on each route** (5 trips): `d |> group_by(route_km) |> slice(1) |> ungroup()`. Fit `models/commute.stan` to these 5 trips, and fit the same model with the vague prior α ~ Normal(0, 100), β ~ Normal(0, 10), σ ~ Normal⁺(0, 50) (copy the file to `models/commute-vague.stan` and edit the priors). Compare the 90% intervals of β and of one 15-km trip. When does the prior matter, and when does it not?

In [ ]:
# d5 <- d |> group_by(route_km) |> slice(1) |> ungroup()

**Your answer:** *(write here)*

### C2. Does the fit recover what generated the data?

The regression generates data, so you can test the whole procedure: choose α = 22, β = 3.5 and σ = 1.5 yourself, simulate 40 trips on her routes, fit `models/commute.stan`, and check whether the 90% intervals contain the values you chose. Repeat with five different seeds. How often would you expect a 90% interval to miss?

In [ ]:
# fake <- d |> select(route_km, x) |> mutate(minutes = rnorm(n(), 22 + 3.5 * x, 1.5))

**Your answer:** *(write here)*